<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 9: Compare Training and Validate a Sound Classifier</h1>
</div>

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

Wayne State University

**Individual graded Lab**

**Version: Student Notebook**

[Open in Colab](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week10/lab09_training_and_validation.ipynb)

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and objectives

How do we choose a trained model without using test results to guide the choice?
Keep the log-mel input and 32-neuron architecture fixed. Compare two learning
rates, use validation loss to choose the checkpoint, and evaluate test afterward.

By the end, you can conduct a controlled comparison, interpret train/validation
curves, use restored weights and distinguish segment from recording predictions.
Complete four checkpoints worth 10 points: CP1=2, CP2=2, CP3=4, CP4=2.
Provided cells run unchanged. Only a small learning-rate/fit block and the
selection block require code edits. This notebook is self-contained.

## Data and evaluation context

The supplied features represent **800 normal MIMII public 1.0 recordings**:
fan, pump, slider and valve; four machine IDs per type; 50 recordings per type/ID.
Each original recording lasts 10 seconds at 16 kHz, retaining source microphone
Channel 1. The source 0 dB condition contains mixed factory background noise.

Each parent recording gives ten nonoverlapping one-second segments. Each segment
is a 32-by-61 log-mel matrix: 32 mel bands and 61 time frames. The numerical log
power reference is digital, not calibrated dB SPL. Extraction is already complete.

Labels are **0=fan, 1=pump, 2=slider, 3=valve**. Machine-type classification is a
course-derived task, not the original MIMII anomalous-sound benchmark.
Only the log-mel values enter the NN. IDs, labels and partition names are metadata.

The fixed same-ID split has 480 training, 160 validation and 160 test recordings,
or 4,800/1,600/1,600 segments. All ten segments from one parent stay together.
The same ID string in different machine types does not identify one physical machine.
Previously used course recordings are not an untouched external benchmark.

**Software/files:** Colab provides TensorFlow/Keras, NumPy, pandas, Matplotlib
and scikit-learn. The supplied feature folder contains `logmel.npz`,
`recordings.csv`, `baseline32_summary.csv`, `FEATURE_DICTIONARY.md` and `LICENSE_DATA.txt`.
No package installation is part of the assessed work.

## Provided setup: run unchanged

The next two cells import packages and locate the supplied files.
This is technical infrastructure, not code you need to reconstruct.
The first learning section then opens the files and inspects what they contain.
No model is trained in this setup block.

In [ ]:
# PROVIDED: a small, reproducible CPU example and familiar plotting tools.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
tf.get_logger().setLevel("ERROR")
tf.config.experimental.enable_op_determinism()

In [ ]:
# PROVIDED: use cached files, or retrieve the supplied course data.
data_base_url = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
data_folder = Path("mimii_logmel800")
data_folder.mkdir(parents=True, exist_ok=True)
feature_file = data_folder / "logmel.npz"
metadata_file = data_folder / "recordings.csv"

In [ ]:
# PROVIDED: download only missing files. Do not edit this setup cell.
from urllib.request import urlretrieve
if not feature_file.is_file():
    urlretrieve(data_base_url + "logmel.npz", feature_file)
if not metadata_file.is_file():
    urlretrieve(data_base_url + "recordings.csv", metadata_file)
reference_file = data_folder / "baseline32_summary.csv"
if not reference_file.is_file():
    urlretrieve(data_base_url + "baseline32_summary.csv", reference_file)
dictionary_file = data_folder / "FEATURE_DICTIONARY.md"
if not dictionary_file.is_file():
    urlretrieve(data_base_url + "FEATURE_DICTIONARY.md", dictionary_file)
license_file = data_folder / "LICENSE_DATA.txt"
if not license_file.is_file():
    urlretrieve(data_base_url + "LICENSE_DATA.txt", license_file)

## First: open the files and understand the data

An `.npz` file is a container of named NumPy arrays. Opening it does not train or
standardize anything. `archive.files` shows the stored names; square brackets
retrieve one named array. The CSV is a separate recording-level metadata table.
Inspect these structures before applying a split or a transformation.

In [ ]:
# PROVIDED: open the feature container and the recording metadata.
archive = np.load(feature_file)
recordings = pd.read_csv(metadata_file)
print("Arrays stored in the NPZ file:", archive.files)
print("Metadata rows and columns:", recordings.shape)

### 1. Read the named arrays

`X` and `y` describe **segments**. `recording_id` and `machine_id` describe
**parent recordings**. `recording_index` links each segment to a parent.
Keep these names separate: an input row is not an independent recording.

In [ ]:
# PROVIDED: extract one named item at a time.
X = archive["X"]
y = archive["y"]
recording_index = archive["recording_index"]
recording_ids = archive["recording_id"]
recording_machine_ids = archive["machine_id"]
class_names = archive["class_names"]

In [ ]:
# PROVIDED: shape tells how many entries each array contains.
print("X shape and dtype:", X.shape, X.dtype)
print("y shape and dtype:", y.shape, y.dtype)
print("recording_index shape:", recording_index.shape)
print("recording_ids shape:", recording_ids.shape)
print("recording_machine_ids shape:", recording_machine_ids.shape)
print("class_names shape:", class_names.shape)

| Array | Shape | Meaning |
|---|---|---|
| X | (8000, 32, 61) | 8000 segments, 32 mel bands, 61 time frames |
| y | (8000,) | One integer machine-type label per segment |
| recording_index | (8000,) | One parent-recording row number per segment |
| recording_ids | (800,) | One stable identifier per parent recording |
| recording_machine_ids | (800,) | One machine ID per parent recording |
| class_names | (4,) | Meaning of label indices 0, 1, 2 and 3 |

`float32` stores numerical log-mel values. `int64` stores integer labels/indices.
A shape `(8000,)` is a one-dimensional array. In X, `X[0]` selects one 32-by-61
matrix; it does not select one frequency band from every segment.

In [ ]:
# PROVIDED: an integer label is an index into class_names.
label_key = pd.DataFrame()
label_key["Label"] = [0, 1, 2, 3]
label_key["Machine type"] = class_names
display(label_key)

### 2. Inspect the recording table

The table has 800 rows, one per 10-second parent recording. A recording ID
identifies a source file. A machine ID identifies a model/ID group within its
machine type. `partition` is the supplied train/validation/test assignment.
These columns support grouping and interpretation; they are not NN inputs.

In [ ]:
# PROVIDED: show a few useful columns, not all source checksums and file paths.
metadata_columns = ["recording_id", "machine_type", "machine_id", "label", "partition"]
metadata_preview = recordings[metadata_columns]
display(metadata_preview.head())

In [ ]:
# PROVIDED: count parent recordings before looking at segment counts.
recording_types = recordings["machine_type"]
metadata_machine_ids = recordings["machine_id"]
type_id_counts = pd.crosstab(recording_types, metadata_machine_ids)
display(type_id_counts)

### 3. Follow one segment back to its parent

The next example inspects the first stored row only to understand file structure.
It is not model evaluation. `recording_index[0]` is the parent-row number.
Using that number in `recording_ids` finds the source recording. No parsing of
filenames is needed. The corresponding class name comes from y, not the ID.

In [ ]:
# PROVIDED: inspect one segment and its parent using explicit variables.
example_segment = X[0]
example_label = y[0]
example_parent_index = recording_index[0]
example_parent_id = recording_ids[example_parent_index]
example_machine_id = recording_machine_ids[example_parent_index]
example_machine_type = class_names[example_label]
print("One segment shape:", example_segment.shape)
print("Label and machine type:", example_label, example_machine_type)
print("Parent row and recording ID:", example_parent_index, example_parent_id)
print("Parent machine ID:", example_machine_id)

### 4. Check why 800 recordings give 8000 segments

Rows 0 through 9 come from one parent recording. Row 10 begins the next parent.
The short preview below shows this relationship. Looking up an array with
several integer indices retrieves one value for each supplied index.
All ten segments from one parent must remain in the same partition.

In [ ]:
# PROVIDED: inspect only twelve rows, rather than printing the full dataset.
segment_rows = np.arange(12)
parent_indices = recording_index[:12]
parent_ids = recording_ids[parent_indices]
segment_labels = y[:12]
lineage_preview = pd.DataFrame()
lineage_preview["Segment row"] = segment_rows
lineage_preview["Parent row"] = parent_indices
lineage_preview["Recording ID"] = parent_ids
lineage_preview["Label"] = segment_labels
display(lineage_preview)

### Provided grouping: recording first

`recording_index` connects each segment to one parent row in `recordings`.
The following indexing copies the parent's partition to every segment. A mask
is a True/False array that selects rows. Do not create a random segment split.

In [ ]:
# PROVIDED: every segment receives its parent recording's partition.
recording_partitions = recordings["partition"].to_numpy()
segment_partitions = recording_partitions[recording_index]
train_mask = segment_partitions == "train"
validation_mask = segment_partitions == "validation"
test_mask = segment_partitions == "test"

In [ ]:
# PROVIDED: each True/False mask selects the matching segment rows.
X_train = X[train_mask]
y_train = y[train_mask]
X_validation = X[validation_mask]
y_validation = y[validation_mask]
X_test = X[test_mask]
y_test = y[test_mask]
print("Segment counts:", len(y_train), len(y_validation), len(y_test))

### One training-global transformation

Compute one mean and one population standard deviation across **all elements of
X_train**. Apply the same pair everywhere. This keeps segment-to-segment level
and contrast differences. It does not make every segment mean 0/SD 1, and it
does not force a Gaussian distribution. `mean()` and `std()` without an axis
combine every element of the supplied array.

In [ ]:
# Fit preprocessing using training values; reuse it on held-out inputs.
training_mean = X_train.mean()
training_std = X_train.std()
X_train_scaled = (X_train - training_mean) / training_std
X_validation_scaled = (X_validation - training_mean) / training_std
X_test_scaled = (X_test - training_mean) / training_std

In [ ]:
# PROVIDED: standardized training statistics should be close to 0 and 1.
print("Training mean and SD used:", training_mean, training_std)
print("Standardized training mean:", X_train_scaled.mean())
print("Standardized training SD:", X_train_scaled.std())
print("Shape after scaling:", X_train_scaled.shape)

## CP1: Keep the comparison controlled

Name the one setting that changes and two settings or data properties that remain fixed. Explain why recording grouping matters.

**Your response:** TODO.

## Guided baseline: learning rate A

The following is the same 32-neuron structure as Lab 8. A checkpoint stores the
weights with the smallest validation loss; saving is provided infrastructure.
`clear_session()` resets Keras naming/state and `set_random_seed(9)` starts the
new model reproducibly. This does not promise identical values on every device.

In [ ]:
# Build the same architecture so the learning-rate comparison stays controlled.
# Clear framework state before constructing a fresh model.
tf.keras.backend.clear_session()
# Keep initialization reproducible for the controlled comparison.
tf.keras.utils.set_random_seed(9)
model_a = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 61)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

In [ ]:
# Configure learning before calling fit.
optimizer_a = tf.keras.optimizers.Adam(learning_rate=0.001)
model_a.compile(optimizer=optimizer_a,
                loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# PROVIDED: store the weights with the lowest validation loss.
baseline_a_file = Path("baseline_a.weights.h5")
baseline_a_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    baseline_a_file, monitor="val_loss", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0
)

In [ ]:
# Train run A and monitor validation without fitting on it.
history_a = model_a.fit(
    X_train_scaled, y_train,
    validation_data=(X_validation_scaled, y_validation),
    epochs=40, batch_size=32, shuffle=True,
    callbacks=[baseline_a_checkpoint], verbose=2
)

The Keras history stores named lists from training.
Read each list into a variable before plotting it. The plots below use epoch
numbers 1 through 40; Python list positions still start at zero.

In [ ]:
# PROVIDED: retrieve each numerical series separately.
training_losses = history_a.history["loss"]
validation_losses = history_a.history["val_loss"]
training_accuracies = history_a.history["accuracy"]
validation_accuracies = history_a.history["val_accuracy"]
epoch_numbers = np.arange(1, 41)

In [ ]:
# PROVIDED: an explicit loss plot, without a plotting helper or loop.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_losses, label="Training")
plt.plot(epoch_numbers, validation_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("A: learning rate 0.001: loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: accuracy is a different metric from loss.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_accuracies, label="Training")
plt.plot(epoch_numbers, validation_accuracies, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Segment accuracy")
plt.title("A: learning rate 0.001: accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## CP2: Run one controlled alternative

Use learning rate **0.0001** for B. Keep one 32-unit hidden layer, the same seed,
batch size 32 and 40 epochs. The architecture and checkpoint setup are supplied.
Complete B's learning rate and fit call using A as the analogous example.

In [ ]:
# Keep architecture and initialization fixed for the alternative run.
# Clear framework state before constructing a fresh model.
tf.keras.backend.clear_session()
# Keep initialization reproducible for the controlled comparison.
tf.keras.utils.set_random_seed(9)
model_b = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 61)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

Complete B's learning rate.

**API syntax hint:**

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=rate)
model.compile(optimizer=optimizer, loss="loss_name", metrics=["accuracy"])
```

In [ ]:
# Change only the requested learning-rate setting.
learning_rate_b = ...
optimizer_b = tf.keras.optimizers.Adam(learning_rate=learning_rate_b)
model_b.compile(optimizer=optimizer_b,
                loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# PROVIDED: store the weights with the lowest validation loss.
alternative_b_file = Path("alternative_b.weights.h5")
alternative_b_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    alternative_b_file, monitor="val_loss", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0
)

Complete the training and validation inputs. Keep the supplied callback.

**API syntax hint:**

```python
history = model.fit(
    inputs, targets, validation_data=(validation_inputs, validation_targets),
    epochs=number_of_epochs, batch_size=batch_size, verbose=2
)
```

`verbose=2` prints one summary per epoch, including training and validation metrics. This changes the display, not the learning settings.

In [ ]:
# Train run B with the same data, budget and checkpoint rule.
history_b = model_b.fit(
    ..., ..., validation_data=...,
    epochs=40, batch_size=32, shuffle=True,
    callbacks=[alternative_b_checkpoint], verbose=2
)

The Keras history stores named lists from training.
Read each list into a variable before plotting it. The plots below use epoch
numbers 1 through 40; Python list positions still start at zero.

In [ ]:
# PROVIDED: retrieve each numerical series separately.
training_losses = history_b.history["loss"]
validation_losses = history_b.history["val_loss"]
training_accuracies = history_b.history["accuracy"]
validation_accuracies = history_b.history["val_accuracy"]
epoch_numbers = np.arange(1, 41)

In [ ]:
# PROVIDED: an explicit loss plot, without a plotting helper or loop.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_losses, label="Training")
plt.plot(epoch_numbers, validation_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("B: learning rate 0.0001: loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: accuracy is a different metric from loss.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_accuracies, label="Training")
plt.plot(epoch_numbers, validation_accuracies, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Segment accuracy")
plt.title("B: learning rate 0.0001: accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: find the position, epoch and loss in separate steps.
validation_losses_a = history_a.history["val_loss"]
validation_losses_b = history_b.history["val_loss"]
best_position_a = np.argmin(validation_losses_a)
best_position_b = np.argmin(validation_losses_b)
best_epoch_a = int(best_position_a) + 1
best_epoch_b = int(best_position_b) + 1
best_loss_a = validation_losses_a[best_position_a]
best_loss_b = validation_losses_b[best_position_b]

In [ ]:
# PROVIDED: fill the comparison table one column at a time.
comparison = pd.DataFrame()
comparison["Run"] = ["A", "B"]
comparison["Learning rate"] = [0.001, learning_rate_b]
comparison["Best epoch"] = [best_epoch_a, best_epoch_b]
comparison["Minimum validation loss"] = [best_loss_a, best_loss_b]
display(comparison)

**CP2 response:** Record the best epoch and validation loss for both runs. Describe one difference in their learning curves.

**Your response:** TODO.

## CP3: Select before test evaluation

Choose the run with the smaller minimum validation loss; if the values tie
exactly, choose A. Use the stored weights of that run, not final weights or a
newly initialized retraining. Loading checkpoints below is supplied infrastructure.
The test block follows your choice and must not be used to revise it.

In [ ]:
# PROVIDED: restore each run's validation-selected weights.
model_a.load_weights(baseline_a_file)
model_b.load_weights(alternative_b_file)

Set selected_model to model_a or model_b and selected_run to "A" or "B". Use validation evidence only.

In [ ]:
# Use the stated validation rule; do not choose using test results.
selected_model = ...
selected_run = ...

**CP3 selection response:** State your selected run and cite both validation losses. Why should you not choose using test accuracy?

**Your response:** TODO.

In [ ]:
# PROVIDED: evaluate only the chosen, restored model.
test_probabilities = selected_model.predict(X_test_scaled, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_segment_accuracy = accuracy_score(y_test, test_predictions)
print("Frozen selected run:", selected_run)
print("Test segment accuracy:", test_segment_accuracy)

### From segments to parent recordings

The supplied arrays retain ten consecutive segments for each parent recording.
The provided loader preserves that order. Reshape groups the ten rows, and
`mean(axis=1)` averages ten probability vectors. Argmax then picks one class per
recording. Do not average nominal class numbers or split a parent across sets.
The aggregation is provided code; focus on the observation unit.

In [ ]:
# PROVIDED: create one group of ten segments for every test recording.
number_of_test_recordings = len(y_test) // 10
test_probability_groups = test_probabilities.reshape(number_of_test_recordings, 10, 4)
test_label_groups = y_test.reshape(number_of_test_recordings, 10)

In [ ]:
# PROVIDED: average probabilities within each parent, then predict its class.
test_recording_probabilities = test_probability_groups.mean(axis=1)
test_recording_labels = test_label_groups[:, 0]
test_recording_predictions = np.argmax(test_recording_probabilities, axis=1)
test_recording_accuracy = accuracy_score(test_recording_labels, test_recording_predictions)
print("Test recordings:", len(test_recording_labels))
print("Test recording accuracy:", test_recording_accuracy)
recording_confusion = confusion_matrix(test_recording_labels, test_recording_predictions, labels=[0, 1, 2, 3])
recording_display = ConfusionMatrixDisplay(recording_confusion, display_labels=class_names)
recording_display.plot()
plt.title("Frozen test recordings: actual rows, predicted columns")
plt.tight_layout()
plt.show()

## CP4: Interpret the engineering result

Report both accuracies, identify one off-diagonal recording error with its count, and state one limitation of deployment claims.

**Your response:** TODO.

## Optional challenge: inspect a probability vector

Inspect one correctly predicted validation/test segment and, if available, one
incorrectly predicted segment. Which two classes have the highest probabilities?
If all predictions are correct, compare two correct examples instead.
No extra model training, submission or points are required.

## Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit this
Markdown cell and change `[ ]` to `[x]`.

- [ ] Enter your name and WSU AccessID.
- [ ] Complete required code and short written responses.
- [ ] Keep the required model/history/metric outputs visible.
- [ ] Restart the runtime and run cells from top to bottom after completing TODOs.
- [ ] Save `Lab09_Firstname_Lastname.ipynb` and its matching PDF.
- [ ] Submit both files through Canvas as instructed. Do not submit through GitHub.

No separate report is required. Optional work is ungraded. PDF rendering issues
receive corrective feedback; the notebook is used to assess obscured evidence.

## References and attribution

- [MIMII public 1.0](https://zenodo.org/records/3384388), DOI 10.5281/zenodo.3384388.
- Purohit et al., [MIMII dataset paper](https://arxiv.org/abs/1909.09347), 2019.
- [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Keras Sequential guide](https://keras.io/guides/sequential_model/).

Original authors: Harsh Purohit, Ryo Tanabe, Kenji Ichige, Takashi Endo,
Yuki Nikaido, Kaori Suefusa and Yohei Kawaguchi (Hitachi, Ltd.).
Course changes: selected normal 0-dB recordings, retained Channel 1, segmented
each recording into ten one-second inputs, computed log-mel features, and added
labels, metadata and a recording-level split. Preserve attribution and ShareAlike
terms when redistributing derived data. See supplied `FEATURE_DICTIONARY.md` and
`LICENSE_DATA.txt`. This is a teaching task, not an official MIMII benchmark.